# CIMIS Data Collection & Graph Generation
This notebook handles the robust retrieval of hourly weather data from the CIMIS API, performs an integrity cross-check, and builds the spatial adjacency matrix for the Graph Neural Network.

In [ ]:
import os
import time
import datetime
import requests
import numpy as np
import pandas as pd
from math import radians, cos, sin, asin, sqrt

# --- 1. CONFIGURATION & IDS ---
CIMIS_APP_KEY = 

# Unified list of all target CIMIS station IDs
TARGET_IDS = [
    106, 103, 158, 83, 77, 144, 187, 157, 213, 178, 170,
    250, 226, 6, 139, 235, 212, 140, 247, 47, 242, 243, 248,
    254, 171, 191, 253, 104, 211,
    195, 13, 228, 227, 262, 131, 84, 70, 249, 71, 194, 206
]

## 1. Weather Data Retrieval API Functions
These functions handle the direct communication with the CIMIS API, ensuring robust timeouts and extracting the nested JSON responses into clean Pandas DataFrames.

In [ ]:
def get_cimis_data_full(app_key, station_ids, start_date, end_date, unit='M'):
    """Fetches ALL available hourly data items from CIMIS."""
    ids_str = ",".join(map(str, station_ids))
    data_items = (
        "hly-air-tmp,hly-dew-pnt,hly-rel-hum,hly-precip,hly-sol-rad,"
        "hly-net-rad,hly-soil-tmp,hly-wind-spd,hly-wind-dir,hly-vap-pres,hly-eto"
    )
    url = (
        f"http://et.water.ca.gov/api/data?appKey={app_key}"
        f"&targets={ids_str}&startDate={start_date}&endDate={end_date}"
        f"&unitOfMeasure={unit}&dataItems={data_items}"
    )
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json'
    }
    try:
        response = requests.get(url, headers=headers, timeout=45)
        if not response.text: return pd.DataFrame()
        data = response.json()
        if isinstance(data, dict) and data.get('Data'):
            records = data['Data']['Providers'][0]['Records']
            return pd.DataFrame(records)
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ Connection Failed: {e}")
        return pd.DataFrame()

def extract_all_features(day_df):
    """Flattens all environmental dictionaries into a clean numeric table."""
    mapping = {
        'HlyAirTmp': 'Temp_C',
        'HlyDewPnt': 'DewPoint_C',
        'HlyRelHum': 'Humidity_pct',
        'HlyPrecip': 'Precip_mm',
        'HlySolRad': 'SolarRad_Wm2',
        'HlySoilTmp': 'SoilTemp_C',
        'HlyWindSpd': 'Wind_ms',
        'HlyWindDir': 'WindDir_deg',
        'HlyEto': 'ETo_mm'
    }
    for api_col, clean_name in mapping.items():
        if api_col in day_df.columns:
            day_df[clean_name] = pd.to_numeric(
                day_df[api_col].apply(lambda x: x.get('Value') if isinstance(x, dict) else None),
                errors='coerce'
            )
    final_cols = ['Station', 'Date', 'Hour'] + list(mapping.values())
    return day_df[final_cols]

## 2. Multi-Year Download Execution
Iterates through the specified time windows safely to respect API limits, saving the data directly into individual `.csv` files for each station.

In [ ]:
import shutil

def download_weather_per_node(app_key, target_ids, start_date_str, end_date_str, output_dir="data/station_weather_files"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Created directory: {output_dir}")

    start_dt = datetime.datetime.strptime(start_date_str, '%Y-%m-%d').date()
    end_dt = datetime.datetime.strptime(end_date_str, '%Y-%m-%d').date()
    
    current_date = start_dt
    while current_date <= end_dt:
        date_str = current_date.strftime('%Y-%m-%d')
        print(f"📅 Syncing {date_str}...", end=" ", flush=True)
        
        try:
            day_df = get_cimis_data_full(app_key, target_ids, date_str, date_str)
            if not day_df.empty:
                processed_df = extract_all_features(day_df)
                unique_stations = processed_df['Station'].unique()
                for station_id in unique_stations:
                    station_data = processed_df[processed_df['Station'] == station_id]
                    file_path = os.path.join(output_dir, f"station_{station_id}.csv")
                    
                    file_exists = os.path.isfile(file_path)
                    station_data.to_csv(file_path, mode='a', index=False, header=not file_exists)
                print(f"✅ Saved {len(unique_stations)} stations.")
            else:
                print("⚠️ No data for this date.")
        except Exception as e:
            print(f"❌ Error on {date_str}: {e}")
            
        current_date += datetime.timedelta(days=1)
        time.sleep(0.5)

# --- MULTI-YEAR EXECUTION BLOCK ---
YEARS = [2020, 2021, 2022, 2023, 2024]
START_MD = "04-01"
END_MD = "07-01"
OUTPUT_FOLDER = "data/station_weather_files"

# Optional: Clear existing folder for a fresh redownload
# if os.path.exists(OUTPUT_FOLDER):
#     shutil.rmtree(OUTPUT_FOLDER)

# Loop through each year
for year in YEARS:
    yearly_start = f"{year}-{START_MD}"
    yearly_end = f"{year}-{END_MD}"
    print(f"\n🚀 --- STARTING DOWNLOAD FOR {year} --- 🚀")
    
    # Uncomment to actually run the download
    # download_weather_per_node(CIMIS_APP_KEY, TARGET_IDS, yearly_start, yearly_end, OUTPUT_FOLDER)

print(f"\n✨ SUCCESS: All data saved individually by station in '{OUTPUT_FOLDER}'.")

## 3. Data Integrity Cross-Check
This cell verifies the downloaded data against our `TARGET_IDS` list to ensure no stations were dropped, and prints out file statistics.

In [ ]:
def cross_check_downloaded_data(target_ids, output_dir="data/station_weather_files"):
    """Cross-checks which files were successfully downloaded and their integrity."""
    print("📊 DATA INTEGRITY CROSS-CHECK")
    print("="*50)
    
    if not os.path.exists(output_dir):
        print(f"❌ Error: {output_dir} directory not found.")
        return
        
    found_files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]
    found_ids = [int(f.split('_')[1].split('.')[0]) for f in found_files]
    
    missing_ids = set(target_ids) - set(found_ids)
    
    print(f"✅ Target Stations: {len(target_ids)}")
    print(f"✅ Downloaded Station Files: {len(found_files)}\n")
    
    if missing_ids:
        print(f"⚠️ MISSING STATIONS ({len(missing_ids)}): {sorted(list(missing_ids))}")
    else:
        print("💎 SUCCESS: All target stations were successfully downloaded!")
        
    print("\nFile Statistics Sample:")
    for f in found_files[:5]:
        df = pd.read_csv(os.path.join(output_dir, f))
        print(f" - {f}: {len(df)} hourly records, {len(df.columns)} features.")

cross_check_downloaded_data(TARGET_IDS)

## 4. Graph Creation (Distance Matrix)
Calculates the Haversine distance between all active CIMIS stations to build the geographical Adjacency Matrix used by the GatedGraphConv layer.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Calculates the distance in kilometers between two points on Earth."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius of earth in kilometers
    return c * r

def save_station_registry(app_key, target_ids):
    """Captures GPS coordinates for the graph creation."""
    print("🛰️ Downloading station GPS metadata...")
    ids_str = ",".join(map(str, target_ids))
    url = f"http://et.water.ca.gov/api/station?appKey={app_key}&targets={ids_str}"
    
    try:
        resp = requests.get(url).json()
        stations = resp.get('Stations', [])
        data = []
        for s in stations:
            lat_val = float(s.get('HmsLatitude', "0/0").split('/')[-1].strip())
            lon_val = float(s.get('HmsLongitude', "0/0").split('/')[-1].strip())
            data.append({
                'ID': int(s['StationNbr']),
                'Name': s['Name'],
                'Lat': lat_val,
                'Lon': lon_val
            })
        registry_df = pd.DataFrame(data)
        registry_df.to_csv('data/station_registry.csv', index=False)
        return registry_df
    except Exception as e:
        print(f"❌ Metadata Failed: {e}")
        return pd.DataFrame()

def build_adjacency_matrix(df, threshold_km=50.0):
    """Creates an N x N matrix where 1 indicates stations are neighbors."""
    n = len(df)
    adj_matrix = np.zeros((n, n))
    df = df.sort_values('ID').reset_index(drop=True)
    
    for i in range(n):
        for j in range(i + 1, n): 
            dist = haversine(
                df.iloc[i]['Lat'], df.iloc[i]['Lon'],
                df.iloc[j]['Lat'], df.iloc[j]['Lon']
            )
            if dist <= threshold_km:
                adj_matrix[i, j] = 1
                adj_matrix[j, i] = 1
                
    return adj_matrix, df['ID'].tolist()

# Execute Graph Creation
registry_df = save_station_registry(CIMIS_APP_KEY, TARGET_IDS)
if not registry_df.empty:
    adj_matrix, sorted_ids = build_adjacency_matrix(registry_df, threshold_km=50.0)
    np.save("data/adjacency_matrix.npy", adj_matrix)
    print("\n📊 GRAPH TOPOLOGY REPORT")
    print("-" * 30)
    print(f"✅ Nodes (Stations): {len(sorted_ids)}")
    print(f"✅ Total Edges: {int(np.sum(adj_matrix)) // 2}")
    print(f"✅ Avg. Neighbors per Node: {np.mean(np.sum(adj_matrix, axis=1)):.1f}")